In [1]:
import os
import math
import random 
import pandas as pd
import numpy as np
import datetime as dt
from pandas_datareader import data as pdr
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from Util_def_nanfix import *
from Util_model import *
from pypfopt import (
    EfficientFrontier,
    risk_models,
    expected_returns,
    objective_functions,
)

import warnings
warnings.filterwarnings('ignore')


Devices:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU details:  {'device_name': 'METAL'}


In [2]:
stock_list = pd.read_excel('US_list.xlsx', sheet_name='nasdaq_nyse_mega')
stock_list = stock_list['SYMBOL'].tolist()

# 5 years data
startDate = dt.datetime(2015, 1, 1)
endDate = dt.datetime(2025, 7, 28)

start_rebalance_year = 2020  # startDate.year + 3

data = getData(stock_list, startDate, endDate)
data.fillna(method='ffill', inplace=True)
data.fillna(method='bfill', inplace=True)
print(data.info())
avg_days = avg_days_per_month(data)

print("=" * 50)
print("Min Date:", data.index.min())
print("Max Date:", data.index.max())
print("Start Rebalance Year:", start_rebalance_year)
print(f"Average number of trading days per month: {avg_days}", "days")
print("=" * 50)


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  59 of 59 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2656 entries, 2015-01-02 to 2025-07-25
Data columns (total 59 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    2656 non-null   float64
 1   AMD     2656 non-null   float64
 2   AMZN    2656 non-null   float64
 3   ASML    2656 non-null   float64
 4   AVGO    2656 non-null   float64
 5   AZN     2656 non-null   float64
 6   COST    2656 non-null   float64
 7   CSCO    2656 non-null   float64
 8   GOOG    2656 non-null   float64
 9   GOOGL   2656 non-null   float64
 10  LIN     2656 non-null   float64
 11  META    2656 non-null   float64
 12  MSFT    2656 non-null   float64
 13  NFLX    2656 non-null   float64
 14  NVDA    2656 non-null   float64
 15  PLTR    2656 non-null   float64
 16  TMUS    2656 non-null   float64
 17  TSLA    2656 non-null   float64
 18  ABBV    2656 non-null   float64
 19  ABT     2656 non-null   float64
 20  AXP     2656 non-null   float64
 21  BABA    2656 non-nu

In [3]:
# --- 2. คลาส RegimeGatedAttention (RGA) ---
class RegimeGatedAttention(layers.Layer):
    """
    Keras Layer สำหรับ Regime-Gated Attention (RGA)
    """
    def __init__(self, d_model, d_regime, cnn_filters, cnn_kernel_size, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.d_regime = d_regime
        self.scale = tf.sqrt(tf.cast(d_model, tf.float32))

        # Step 1: Regime State Encoder (CNN)
        self.regime_cnn = layers.Conv1D(
            filters=cnn_filters,
            kernel_size=cnn_kernel_size,
            activation='relu',
            padding='causal'
        )
        self.regime_pool = layers.GlobalAveragePooling1D()
        self.regime_mlp = layers.Dense(d_regime, activation='relu')

        # Step 2: Standard Q, K, V Projections
        self.W_q = layers.Dense(d_model, use_bias=False)
        self.W_k = layers.Dense(d_model, use_bias=False)
        self.W_v = layers.Dense(d_model, use_bias=False)

        # Step 3: FiLM Gating Layers
        self.film_gamma_q = layers.Dense(d_model)
        self.film_beta_q = layers.Dense(d_model)
        self.film_gamma_k = layers.Dense(d_model)
        self.film_beta_k = layers.Dense(d_model)

    def call(self, inputs):
        x_features, r_prices = inputs  # x: (B, N, D_model), r: (B, T, N)

        # STEP 1: สร้าง Regime Vector (S_t)
        s = self.regime_cnn(r_prices) 
        s = self.regime_pool(s) 
        s_t = self.regime_mlp(s) 

        # STEP 2: สร้าง Q, K, V
        q = self.W_q(x_features)
        k = self.W_k(x_features)
        v = self.W_v(x_features)

        # STEP 3: การ Gating (FiLM)
        gamma_q = self.film_gamma_q(s_t)
        beta_q = self.film_beta_q(s_t)
        gamma_k = self.film_gamma_k(s_t)
        beta_k = self.film_beta_k(s_t)

        gamma_q = tf.expand_dims(gamma_q, axis=1)
        beta_q = tf.expand_dims(beta_q, axis=1)
        gamma_k = tf.expand_dims(gamma_k, axis=1)
        beta_k = tf.expand_dims(beta_k, axis=1)

        q_prime = (q * (1 + gamma_q)) + beta_q
        k_prime = (k * (1 + gamma_k)) + beta_k

        # STEP 4: คำนวณ Attention
        attn_scores = tf.matmul(q_prime, k_prime, transpose_b=True) / self.scale
        attn_weights = tf.nn.softmax(attn_scores, axis=-1)
        output = tf.matmul(attn_weights, v)

        return output, attn_weights

    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model': self.d_model,
            'd_regime': self.d_regime,
            'cnn_filters': self.regime_cnn.filters,
            'cnn_kernel_size': self.regime_cnn.kernel_size[0],
        })
        return config

# --- 3. ฟังก์ชันเตรียมข้อมูล ---

def create_features_and_targets(price_data, feature_windows, h_forecast):
    """
    สร้างฟีเจอร์ (Momentum, Volatility) และ Targets (Future Returns)
    """
    returns = price_data.pct_change()
    
    feature_list = []
    # สร้างฟีเจอร์ Momentum และ Volatility
    for window in feature_windows:
        momentum = returns.rolling(window).mean()
        volatility = returns.rolling(window).std()
        feature_list.append(momentum)
        feature_list.append(volatility)
        
    # Stack ฟีเจอร์: (Time, N_Assets, D_Model)
    features_stacked = np.stack([f.values for f in feature_list], axis=2)
    
    # สร้าง Target: ผลตอบแทนในอนาคต h_forecast วันข้างหน้า
    target_returns = price_data.pct_change(h_forecast).shift(-h_forecast)
    
    return features_stacked, target_returns

def create_window_samples(price_data, features, targets, t_lookback, h_forecast, dates_index):
    """
    สร้างชุดข้อมูลแบบ Sliding Window
    """
    n_samples = len(price_data)
    n_assets = price_data.shape[1]
    d_model = features.shape[2]

    # หาจุดเริ่มต้นที่ข้อมูลไม่เป็น NaN (หลังจาก rolling window ที่ยาวที่สุด)
    start_idx = np.max([t_lookback] + FEATURE_WINDOWS)
    # หาจุดสิ้นสุด (ก่อนที่ target จะเป็น NaN)
    end_idx = n_samples - h_forecast
    
    # Pre-allocate-Memory
    X_prices = np.empty((end_idx - start_idx, t_lookback, n_assets))
    X_features = np.empty((end_idx - start_idx, n_assets, d_model))
    y_targets = np.empty((end_idx - start_idx, n_assets))
    sample_dates = []

    print(f"กำลังสร้าง {end_idx - start_idx} samples...")
    
    price_data_np = price_data.values
    
    idx_counter = 0
    for t in range(start_idx, end_idx):
        X_prices[idx_counter] = price_data_np[t - t_lookback : t, :]
        X_features[idx_counter] = features[t, :, :]
        y_targets[idx_counter] = targets.iloc[t].values
        sample_dates.append(dates_index[t])
        idx_counter += 1
        
    # === START: FIX 1 (กรอง NaN) ===
    # ตัดส่วนที่ Pre-allocate แต่ไม่ได้ใช้ออก
    X_prices = X_prices[:idx_counter]
    X_features = X_features[:idx_counter]
    y_targets = y_targets[:idx_counter]
    
    # สร้าง mask เพื่อหาแถว (sample) ที่ *ไม่มี* nan ใน y_targets
    valid_mask = ~np.isnan(y_targets).any(axis=1)
    
    X_prices = X_prices[valid_mask]
    X_features = X_features[valid_mask]
    y_targets = y_targets[valid_mask]
    
    # กรอง sample_dates ด้วย
    sample_dates = [d for i, d in enumerate(sample_dates) if valid_mask[i]]
    
    print(f"กรอง sample ที่มี nan ใน target: เหลือ {len(y_targets)} samples")
    # === END: FIX 1 ===

    # Standardize ข้อมูล (สำคัญมาก!)
    scaler_p = StandardScaler()
    X_prices = scaler_p.fit_transform(X_prices.reshape(-1, n_assets)).reshape(X_prices.shape)
    
    scaler_f = StandardScaler()
    X_features = scaler_f.fit_transform(X_features.reshape(-1, d_model)).reshape(X_features.shape)

    return X_prices, X_features, y_targets, sample_dates, (scaler_p, scaler_f)

# --- 4. ฟังก์ชัน Loss (Negative Sharpe Ratio) ---

def negative_sharpe_loss(y_true, y_pred):
    """
    Loss function: Negative Sharpe Ratio (ฉบับแก้ไข)
    y_true: (B, N) - ผลตอบแทนจริงในอนาคต (Actual future returns)
    y_pred: (B, N) - น้ำหนักพอร์ตที่โมเดลทำนาย (Predicted portfolio weights)
    """
    portfolio_returns = tf.reduce_sum(tf.multiply(y_pred, y_true), axis=1)
    
    mean_return = tf.reduce_mean(portfolio_returns)
    std_return = tf.math.reduce_std(portfolio_returns)
    
    # === START: FIX 2 (ทำให้ Loss เสถียร) ===
    # ใช้ tf.maximum เพื่อ "หนีบ" ค่า std ไม่ให้ต่ำกว่า 1e-6
    stable_std_return = tf.maximum(std_return, 1e-6)
    # === END: FIX 2 ===

    sharpe_ratio = mean_return / stable_std_return
    
    return -sharpe_ratio

In [4]:
T_LOOKBACK = 60
H_FORECAST = 21
FEATURE_WINDOWS = [5, 10, 21, 60]

D_REGIME = 16
CNN_FILTERS = 32
CNN_KERNEL = 5


if data.empty or data.shape[1] == 0: # ตรวจสอบว่ามีข้อมูลเหลือหรือไม่
    print("ไม่สามารถดึงข้อมูลได้ หรือ ไม่มีหุ้นใดมีข้อมูลครบถ้วน, จบการทำงาน")
else:
    N_ASSETS = data.shape[1]
    
    features, targets = create_features_and_targets(data, FEATURE_WINDOWS, H_FORECAST)
    
    D_MODEL_AUTO = features.shape[2]
    print(f"N_Assets (หลังกรอง): {N_ASSETS}")
    print(f"D_model (คำนวณจากฟีเจอร์): {D_MODEL_AUTO}")
    
    X_prices, X_features, y_targets, sample_dates, scalers = create_window_samples(
        data, features, targets, T_LOOKBACK, H_FORECAST, data.index
    )
    
    if len(y_targets) == 0:
            print("ไม่มี sample เหลือหลังจากการกรอง NaN, จบการทำงาน")
            exit()

    # --- C. แบ่งข้อมูล Train/Test ---
    split_date = f"{start_rebalance_year}-01-01"
    sample_dates_df = pd.Series(sample_dates)
    
    train_mask = (sample_dates_df < split_date)
    test_mask = (sample_dates_df >= split_date)

    X_train_p, X_test_p = X_prices[train_mask], X_prices[test_mask]
    X_train_f, X_test_f = X_features[train_mask], X_features[test_mask]
    y_train, y_test = y_targets[train_mask], y_targets[test_mask]

    print(f"Train samples: {len(y_train)}")
    print(f"Test samples:  {len(y_test)}")
    
    if len(y_train) == 0 or len(y_test) == 0:
        print("ข้อมูล Train หรือ Test ว่างเปล่า, กรุณาตรวจสอบช่วงวันที่หรือข้อมูลดิบ")
        exit()

    # --- D. สร้างโมเดล RGA ---
    feature_input = layers.Input(shape=(N_ASSETS, D_MODEL_AUTO), name="Asset_Features")
    price_input = layers.Input(shape=(T_LOOKBACK, N_ASSETS), name="Price_History")

    rga_layer = RegimeGatedAttention(
        d_model=D_MODEL_AUTO,
        d_regime=D_REGIME,
        cnn_filters=CNN_FILTERS,
        cnn_kernel_size=CNN_KERNEL,
        name="RGA_Layer"
    )
    
    rga_output, _ = rga_layer([feature_input, price_input])

    # --- E. สร้าง Output (Portfolio Weights) ---
    asset_scores = layers.Dense(1, activation='relu')(rga_output)
    asset_scores_flat = layers.Flatten()(asset_scores)
    portfolio_weights = layers.Softmax(name="Portfolio_Weights")(asset_scores_flat)

    model = Model(
        inputs=[feature_input, price_input],
        outputs=portfolio_weights,
        name="RGA_Portfolio_Model"
    )

    # --- F. Compile และ Train ---
    
    # === START: FIX 3 (Gradient Clipping) ===
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
        loss=negative_sharpe_loss
    )
    # === END: FIX 3 ===
    
    model.summary()
    
    X_train = [X_train_f, X_train_p]
    X_test = [X_test_f, X_test_p]

    print("\n--- เริ่มต้นการเทรนโมเดล ---")
    
    history = model.fit(
        X_train,
        y_train,
        epochs=10,
        batch_size=32,
        validation_data=(X_test, y_test),
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
    )
    
    print("\n--- การเทรนเสร็จสิ้น ---")

    # --- G. ประเมินผล (ตัวอย่าง) ---
    print("ประเมินผลบน Test Set (Negative Sharpe):")
    test_loss = model.evaluate(X_test, y_test)
    print(f"Test Loss (Negative Sharpe): {test_loss:.4f}")
    print(f"Test Sharpe (approx): {-test_loss:.4f}")

N_Assets (หลังกรอง): 59
D_model (คำนวณจากฟีเจอร์): 8
กำลังสร้าง 2575 samples...
กรอง sample ที่มี nan ใน target: เหลือ 2575 samples
Train samples: 1198
Test samples:  1377


2025-11-01 19:27:54.355654: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-11-01 19:27:54.355681: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-11-01 19:27:54.355685: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-11-01 19:27:54.355696: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-01 19:27:54.355703: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "RGA_Portfolio_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Asset_Features      │ (None, 59, 8)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Price_History       │ (None, 60, 59)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ RGA_Layer           │ [(None, 59, 8),   │     10,736 │ Asset_Features[0… │
│ (RegimeGatedAttent… │ (None, 59, 59)]   │            │ Price_History[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 59, 1)     │          9 │ RGA_Layer[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 59)        │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Portfolio_Weights   │ (None, 59)        │          0 │ flatten[0][0]     │
│ (Softmax)           │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,745 (41.97 KB)

 Trainable params: 10,745 (41.97 KB)

 Non-trainable params: 0 (0.00 B)


--- เริ่มต้นการเทรนโมเดล ---
Epoch 1/10


2025-11-01 19:27:54.996272: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: -0.4627 - val_loss: -19.8189
Epoch 2/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: -0.4472 - val_loss: -20.8354
Epoch 3/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: -0.4280 - val_loss: -21.7511
Epoch 4/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: -0.5206 - val_loss: -23.5314
Epoch 5/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: -0.4981 - val_loss: -22.7824
Epoch 6/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: -0.5110 - val_loss: -22.9955
Epoch 7/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: -0.5321 - val_loss: -23.2261
Epoch 8/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: -0.5103 - val_loss: -31.7303
Epoch 9/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: -0.5136 - val_loss: -31.8828
Epoch 10/10
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: -0.5497 - val_loss: -27.8933

--- การเทรนเสร็จสิ้น ---
ประเมินผลบน Test Set (Negative Sharpe):
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: -1.9147
Test Loss